# Task 3.3 — TTS Synthesis in Maithili


In [1]:
!pip install -q transformers accelerate soundfile torchaudio numpy

In [2]:
import gc, json
from pathlib import Path
import numpy as np
import soundfile as sf
import torch
import torchaudio

TARGET_SR_TTS = 22050
TARGET_SR_PROSODY = 16000
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## Load translated text

In [3]:
trans_path = 'translated_maithili.json'
if Path(trans_path).exists():
    data = json.load(open(trans_path, 'r', encoding='utf-8'))
    segments = data.get('segments', []) if isinstance(data, dict) else data
else:
    print(f'Warning: {trans_path} not found, using placeholder')
    segments = [{'maithili_text': 'ई एकटा परीक्षण वाक्य अछि।'}]

print(f'Segments to synthesise: {len(segments)}')

Segments to synthesise: 24


## Synthesise with MMS-TTS on GPU

In [4]:
from transformers import VitsModel, AutoTokenizer

MODEL_NAME = 'facebook/mms-tts-mai'
print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tts_model = VitsModel.from_pretrained(MODEL_NAME).to(device)
tts_model.eval()
print('MMS-TTS loaded!')

audio_segments = []
total = len(segments)

for i, seg in enumerate(segments):
    text = seg.get('maithili_text', seg.get('text', '')).strip()
    if not text:
        continue

    try:
        inputs = tokenizer(text, return_tensors='pt').to(device)
        with torch.no_grad():
            output = tts_model(**inputs)
            waveform = output.waveform.squeeze(0).cpu()
        audio_segments.append(waveform)
    except Exception as e:
        print(f'  Warning segment {i}: {e}')
        audio_segments.append(torch.zeros(int(TARGET_SR_TTS * 0.5)))

    if (i+1) % 5 == 0 or i == total - 1:
        print(f'  Synthesised {i+1}/{total}')
    gc.collect()

del tts_model, tokenizer
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print(f'Generated {len(audio_segments)} audio segments')

Loading facebook/mms-tts-mai...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/47.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

MMS-TTS loaded!
  Synthesised 5/24
  Synthesised 10/24
  Synthesised 15/24
  Synthesised 20/24
  Synthesised 24/24
Generated 24 audio segments


## Concatenate with crossfade

In [5]:
def crossfade_concat(segments_list, sr, crossfade_ms=50):
    """Concatenate with short crossfade + silence gap."""
    crossfade_samples = int(crossfade_ms / 1000 * sr)
    silence = torch.zeros(int(0.25 * sr))  # 250ms gap
    parts = []
    for i, seg in enumerate(segments_list):
        if i > 0 and crossfade_samples > 0 and len(parts) > 0:
            parts.append(silence)
        parts.append(seg)
    return torch.cat(parts)

flat_audio = crossfade_concat(audio_segments, TARGET_SR_TTS)
duration = flat_audio.numel() / TARGET_SR_TTS
print(f'Flat synthesis: {duration:.1f}s ({duration/60:.1f} min)')

sf.write('output_LRL_flat.wav', flat_audio.numpy(), TARGET_SR_TTS)
print('Saved output_LRL_flat.wav')

Flat synthesis: 540.6s (9.0 min)
Saved output_LRL_flat.wav
